> **公开学习副本**：保留教学代码与解释，已清除运行输出、执行计数、嵌入附件及个人元数据。完整执行原件和H02交接包保留在本地/组内私有空间。不要把本副本误认为团队已验收的新数据版本。
>
> 数据来自Olist；基础教学材料参考[CercaTrovato/olist-repurchase-analysis](https://github.com/CercaTrovato/olist-repurchase-analysis)，保留上游代码与数据许可。仅在本地有原始数据时运行；输出可能包含逐行客户数据，保持在被Git忽略的outputs目录。源码中的定义及限制仍应阅读。
> 历史学习课：用于理解问题，不取代当前H01/H02。窗口对比只针对已开放的三期，不要用它探索待封存的最终标签。

# 九表关系 → 时间 → 质量：一次可从头运行的核查

本文件自带初始化，不依赖其他notebook里的orders或frames。按“重新启动内核并运行全部”即可复现。
只检查、记录；不删除、不填补、不规定购买资格或预测窗口。
CSV不带数据库外键约束，下文验证的是当前文件中的键及逻辑关联。
数据来源：小组仓库分发的Olist原始ZIP，CC BY-NC-SA 4.0。
https://github.com/CercaTrovato/olist-repurchase-analysis


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import pandas as pd

candidates = [p / 'data/olist-v2.zip' for p in [Path.cwd(), *Path.cwd().parents]]
DATA = next((p for p in candidates if p.exists()), None)
assert DATA is not None, '请把原始ZIP放入notebook旁的data文件夹'
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == '967e41e04fc306fe604e2a693f488995a8b41e5047418f8a5c8e4abd6deca784'
names = {
    'orders':'olist_orders_dataset.csv', 'customers':'olist_customers_dataset.csv',
    'items':'olist_order_items_dataset.csv', 'payments':'olist_order_payments_dataset.csv',
    'reviews':'olist_order_reviews_dataset.csv', 'products':'olist_products_dataset.csv',
    'sellers':'olist_sellers_dataset.csv', 'geo':'olist_geolocation_dataset.csv',
    'translation':'product_category_name_translation.csv'
}
frames = {}
with ZipFile(DATA) as z:
    for name, file in names.items():
        with z.open(file) as f:
            frames[name] = pd.read_csv(f, dtype={
                'customer_zip_code_prefix':'string', 'seller_zip_code_prefix':'string',
                'geolocation_zip_code_prefix':'string'
            })
print('原包校验通过：', DATA)
print(pd.DataFrame([{'table': n, 'rows':len(t), 'columns':t.shape[1]}
                    for n,t in frames.items()]).to_string(index=False))


## 1. 键与整行重复
候选键必须非空且唯一。复合键需要一起检查。CSV中不设定数据库主键，只记录当前观察证据。


In [ ]:
key_specs=[
    ('orders',['order_id']), ('customers',['customer_id']),
    ('customers',['customer_unique_id']), ('items',['order_id','order_item_id']),
    ('payments',['order_id','payment_sequential']), ('reviews',['review_id']),
    ('reviews',['review_id','order_id']), ('products',['product_id']),
    ('sellers',['seller_id']), ('geo',['geolocation_zip_code_prefix']),
    ('translation',['product_category_name'])
]
keys=[]
for name, cols in key_specs:
    t=frames[name]
    keys.append({'table':name,'key':' + '.join(cols),
                 'null_rows':int(t[cols].isna().any(axis=1).sum()),
                 'duplicate_key_rows_beyond_first':int(t.duplicated(cols).sum())})
print(pd.DataFrame(keys).to_string(index=False))
duplicates=pd.DataFrame([{'table':n,'full_duplicate_rows_beyond_first':int(t.duplicated().sum())}
                         for n,t in frames.items()])
print('\n整行重复：')
print(duplicates.to_string(index=False))


**解释：** 商品与支付的复合键通过。review_id有814个超出首次出现的重复，但(review_id,order_id)组合无重复，不能据此随意删掉重复review_id。geo前缀不是唯一键；地理表有261831个超出首次出现的整行重复。其余八表没有整行重复。customer_unique_id本来用于跨订单识别买家，不要求在客户资料表唯一。


## 2. 九条关联逐条检查
同时检查：父端键是否唯一、子端空键、非空但无父记录、父端无子记录。
`无子记录`不一定是错误，需根据将来的分析用途决定影响。


In [ ]:
relation_specs=[
    ('orders→customers','orders','customer_id','customers','customer_id'),
    ('items→orders','items','order_id','orders','order_id'),
    ('payments→orders','payments','order_id','orders','order_id'),
    ('reviews→orders','reviews','order_id','orders','order_id'),
    ('items→products','items','product_id','products','product_id'),
    ('items→sellers','items','seller_id','sellers','seller_id'),
    ('products→translation','products','product_category_name','translation','product_category_name'),
    ('customers→geo','customers','customer_zip_code_prefix','geo','geolocation_zip_code_prefix'),
    ('sellers→geo','sellers','seller_zip_code_prefix','geo','geolocation_zip_code_prefix')
]
relations=[]
for label, cn, ck, pn, pk in relation_specs:
    child,parent=frames[cn],frames[pn]
    orphan=child[ck].notna() & ~child[ck].isin(parent[pk].dropna())
    parent_ids=parent[pk].dropna().drop_duplicates()
    relations.append({'relation':label,'parent_key_unique':parent[pk].is_unique,
        'child_null_rows':int(child[ck].isna().sum()),
        'unmatched_nonnull_rows':int(orphan.sum()),
        'unmatched_distinct_values':int(child.loc[orphan,ck].nunique()),
        'parent_keys_without_children':int((~parent_ids.isin(child[ck].dropna())).sum())})
relation_result=pd.DataFrame(relations)
print(relation_result.to_string(index=False))


**解读与连接结论：**

| 关联 | 检查结论 | 连接时的处理边界 |
|---|---|---|
| orders→customers | 全部匹配；两侧customer_id在本文件都唯一 | 可按1:1连接；不把customer_id误当跨订单买家ID |
| items→orders | 明细均找到订单；775笔订单无明细 | 订单到明细1:N；金额未知不自动填0 |
| payments→orders | 支付均找到订单；1笔订单无支付 | 订单到支付1:N |
| reviews→orders | 评价均找到订单；768笔订单无评价 | 订单到评价1:N；无评价不等于差评 |
| items→products | 全匹配 | 明细到商品N:1，可用many_to_one验证 |
| items→sellers | 全匹配 | 明细到卖家N:1 |
| products→translation | 610商品原始品类缺失；13商品的非空品类无翻译，涉及2种品类 | 保留原始品类；区分两类缺失 |
| customers→geo | 278条资料无地理匹配，涉及157个前缀；父端不唯一 | 不能直接当N:1连接 |
| sellers→geo | 7卖家无地理匹配；父端不唯一 | 同上，按需要决定邮编映射 |

客户/卖家先按邮编连原始geo可能造成大量行膨胀，不为验证关系而执行全量多对多连接。
如果做一行一订单的分析，应先分别汇总items/payments/reviews，再连orders；评价如何汇总还需业务定义。


## 3. 时间字段：原始缺失、解析失败、范围
缺失与解析失败分开报告；成功解析不表示业务含义无误。


In [ ]:
date_specs={
    'orders':['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
              'order_delivered_customer_date','order_estimated_delivery_date'],
    'items':['shipping_limit_date'],
    'reviews':['review_creation_date','review_answer_timestamp']
}
date_values={}; date_rows=[]
for name, cols in date_specs.items():
    for col in cols:
        raw=frames[name][col]
        parsed=pd.to_datetime(raw,errors='coerce')
        date_values[(name,col)]=parsed
        date_rows.append({'table':name,'field':col,'raw_missing':int(raw.isna().sum()),
            'parse_fail_nonnull':int((raw.notna()&parsed.isna()).sum()),
            'min':str(parsed.min()),'max':str(parsed.max())})
print(pd.DataFrame(date_rows).to_string(index=False))


**解释：** 这8个时间字段没有发现非空值的解析失败。订单批准时间缺失160，交承运方时间缺失1783，客户送达时间缺失2965。预计送达日期可能晚于最后下单日，因为它是预计日期，不是新订单发生日期。


In [ ]:
purchase=date_values[('orders','order_purchase_timestamp')]
approval=date_values[('orders','order_approved_at')]
months=pd.period_range(purchase.min().to_period('M'),purchase.max().to_period('M'),freq='M')
monthly=pd.DataFrame({
    'purchase_count':purchase.groupby(purchase.dt.to_period('M')).size(),
    'approval_count':approval.groupby(approval.dt.to_period('M')).size()
}).reindex(months).fillna(0).astype(int)
print('全部月份，尚未按任何订单资格筛选：')
print(monthly.to_string())


**解释：** 原始下单记录2016-09至2018-10；头尾月份稀疏。2018-09下单16笔、有批准时间且批准发生在9月的1笔；10月下单4笔、批准0笔。这是两个事件的统计，不是两份结果冲突。仅凭最大日期不能认证观察完整，本课不据此直接锁定训练/测试日期。


## 4. 时间顺序线索：先标记，再核查
比较时同时报告两列都非空的行数；缺失行不是“通过顺序检查”。不同异常可能出现在同一订单，异常数不能直接相加得到异常订单数。


In [ ]:
o=frames['orders']
sequence_specs=[
    ('批准早于下单','order_approved_at','order_purchase_timestamp'),
    ('交承运方早于下单','order_delivered_carrier_date','order_purchase_timestamp'),
    ('送达早于下单','order_delivered_customer_date','order_purchase_timestamp'),
    ('交承运方早于批准','order_delivered_carrier_date','order_approved_at'),
    ('送达早于交承运方','order_delivered_customer_date','order_delivered_carrier_date')
]
sequence_rows=[]; suspicious=pd.Series(False,index=o.index)
for label, left, right in sequence_specs:
    a=date_values[('orders',left)]; b=date_values[('orders',right)]
    comparable=a.notna()&b.notna(); flag=comparable&a.lt(b)
    suspicious=suspicious|flag
    sequence_rows.append({'check':label,'comparable_rows':int(comparable.sum()),'flagged_rows':int(flag.sum())})
print(pd.DataFrame(sequence_rows).to_string(index=False))
print('至少命中一个上述时间检查的不同订单数：',int(suspicious.sum()))
print('\n前5条待核查记录：')
print(o.loc[suspicious,['order_id','order_status','order_purchase_timestamp','order_approved_at',
                        'order_delivered_carrier_date','order_delivered_customer_date']].head(5).to_string(index=False))


**解释：** 发现166笔“交承运方早于下单”、1359笔“交承运方早于批准”、23笔“送达早于交承运方”。这些是核查线索，不是已经证明的错误，也不代表直接删除：需要查看记录、时间语义和来源机制。如果未来计算物流耗时，应另存标记，并根据目标决定能否使用对应记录。


## 5. 缺失、状态与数值检查
区别数据结构必需键的缺失和可选字段缺失。例如没有评价正文，不代表没有评分或差评。


In [ ]:
missing=[]
for name,t in frames.items():
    for col in t.columns:
        n=int(t[col].isna().sum())
        if n:missing.append({'table':name,'column':col,'missing':n,'rate':n/len(t)})
print(pd.DataFrame(missing).to_string(index=False))
print('\n订单状态：')
print(o['order_status'].value_counts(dropna=False).to_string())
print('\n评价分数：')
print(frames['reviews']['review_score'].value_counts(dropna=False).sort_index().to_string())

numeric_checks=[('items','price'),('items','freight_value'),('payments','payment_value'),
                ('payments','payment_installments'),('products','product_weight_g'),
                ('products','product_length_cm'),('products','product_height_cm'),('products','product_width_cm')]
numeric_rows=[]
for name,col in numeric_checks:
    s=frames[name][col]
    numeric_rows.append({'table':name,'column':col,'min':s.min(),'median':s.median(),'max':s.max(),
                         'negative':int(s.lt(0).sum()),'zero':int(s.eq(0).sum()),'missing':int(s.isna().sum())})
print('\n选定数值字段：')
print(pd.DataFrame(numeric_rows).to_string(index=False))


**解释：** 商品明细价格最小0.85；383条运费为0，可能免费配送，不能自动删除。支付有9条金额为0、2条分期期数为0，应检查原记录及来源。标题缺失87656条、正文缺失58247条，不影响“该行有评分”这个事实；不在本轮填成负面文本。商品多个属性缺失610条，尺寸重量各缺失2条，后续处理取决于是否使用这些特征。


## 6. 金额对账：先各自汇总到订单，再比较
分别汇总，避免明细×支付的重复膨胀。只在两边都有记录的订单上比较，不把没有支付/明细的订单填0。
差额是“商品金额+运费−支付金额”。本轮以绝对差额大于0.01作为核查阈值，不作为自动删除规则。


In [ ]:
items=frames['items']; payments=frames['payments']
goods=items.groupby('order_id').agg(goods=('price','sum'),freight=('freight_value','sum'))
paid=payments.groupby('order_id')['payment_value'].sum().rename('payment')
reconciliation=goods.join(paid,how='inner',validate='one_to_one')
reconciliation['delta']=(reconciliation['goods']+reconciliation['freight']-reconciliation['payment']).round(2)
flag=reconciliation['delta'].abs().gt(0.01)
print('双方均有记录、可对账订单数：',len(reconciliation))
print('差额绝对值大于0.01的订单数：',int(flag.sum()))
print('最大差额绝对值：',reconciliation['delta'].abs().max())
print('差额最大的5笔：')
print(reconciliation.loc[flag].assign(abs_delta=lambda t:t['delta'].abs()).nlargest(5,'abs_delta').round(2).to_string())


**解释：** 98665笔订单两边都有记录，其中303笔差额绝对值大于0.01，最大182.81。不能直接认定是脏数据：先核查支付、运费、舍入及记录覆盖等可能原因，数据未说明的机制不猜成事实。

## 7. 本轮决定：保留原始数据，建立问题清单

现在已完成九条关联和一轮时间/质量检查。没有筛出“合格订单”，也没有定业务目标或90天窗口。

| 问题 | 后续行动 |
|---|---|
| geo重复前缀、重复行 | 如需要地理分析，再决定邮编层级映射；不直接全量拼接 |
| review_id和每单评价不唯一 | 如需要订单评分，先核查评价机制，再确定选取/汇总规则 |
| 订单缺少子表记录 | 用覆盖标记保留；业务目标确定后决定纳入范围 |
| 日期顺序线索 | 查看样例、核查语义；计算耗时前标记 |
| 边界月份稀疏 | 查询来源覆盖，按目标所需后续观察安排时点 |
| 属性/文本缺失 | 根据字段用途决定；若拟合填补器只在训练数据拟合 |
| 金额不一致 | 单独跟踪对账差额，核查前不自动删改 |

下一步是依据这些事实选业务用途和分析单位，再定预测时点、目标定义、可用特征与标签观察期。那些选择明确后才设计时间划分和baseline。
